# dscr-tools — Loan Amortization & DSCR Toolkit
## Demo: Commercial Real Estate Loan Analysis

This notebook walks through a complete CRE loan analysis:
- Amortization schedules across multiple interest methods
- Period-by-period DSCR tracking and covenant monitoring
- Maximum loan sizing via DSCR, LTV, and Debt Yield tests
- Rate shock and NOI stress sensitivity analysis


In [ ]:
import sys
sys.path.insert(0, '..')

from dscrtools import LoanParams, NOISchedule, INTEREST_METHODS, DSCR_BENCHMARKS
from dscrtools.models import amortization, dscr, sizing, stress
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)


## 1. Define the Loan

A $8MM commercial real estate loan with 24 months of interest-only
followed by 30-year amortization. Common structure for value-add deals.


In [ ]:
loan = LoanParams(
    loan_amount=8_000_000,
    interest_rate=0.065,
    amortization_years=30,
    loan_term_years=10,
    interest_method="partial_io",
    io_periods=24,
    payments_per_year=12,
    property_name="Midtown Office Building",
)

print(loan)
print(f"\nAvailable interest methods:")
for method, desc in INTEREST_METHODS.items():
    print(f"  {method:<22} — {desc}")


## 2. Amortization Schedule

Full period-by-period breakdown showing I/O periods transitioning
to full P&I amortization at month 25.


In [ ]:
df = amortization.summary(loan)
print(f"\nTotal interest paid over loan term: \${df['interest'].sum():,.0f}")
print(f"Balloon balance at maturity:        \${df['ending_balance'].iloc[-1]:,.0f}")


## 3. Compare Interest Methods

How does switching from partial I/O to full amortization affect payments?


In [ ]:
methods = ["fixed_30_360", "fixed_actual_360", "interest_only", "partial_io"]

print(f"{'Method':<25} {'Monthly Pmt (Y1)':<20} {'Total Interest':<20} {'Balloon'}")
print("-" * 80)

for method in methods:
    test_loan = LoanParams(
        loan_amount=8_000_000,
        interest_rate=0.065,
        amortization_years=30,
        loan_term_years=10,
        interest_method=method,
        io_periods=24 if method == "partial_io" else 0,
        payments_per_year=12,
    )
    df = amortization.to_dataframe(test_loan)
    first_pmt = df["payment"].iloc[0]
    total_int = df["interest"].sum()
    balloon   = df["ending_balance"].iloc[-1]
    print(f"{method:<25} \${first_pmt:<19,.0f} \${total_int:<19,.0f} \${balloon:,.0f}")


## 4. Define the NOI Schedule

Net Operating Income for a stabilized office property with 3% annual
growth, 5% vacancy, and $10k annual capex reserve.


In [ ]:
noi = NOISchedule(
    base_noi=750_000,
    growth_rate=0.03,
    vacancy_rate=0.05,
    capex_reserve=10_000,
)

print("NOI Projection:")
print(f"{'Year':<8} {'Gross NOI':<15} {'Effective NOI'}")
print("-" * 35)
for yr in range(1, 11):
    print(f"  {yr:<6} \${noi.get_noi(yr):<14,.0f} \${noi.get_effective_noi(yr):,.0f}")


## 5. DSCR Analysis

Period-by-period DSCR tracking against the 1.25x covenant minimum.


In [ ]:
results = dscr.analyze(loan, noi, min_dscr=1.25)
df_dscr = dscr.summary_table(results)

breaches = sum(1 for r in results if r.covenant_breach)
defaults = sum(1 for r in results if r.default)
min_dscr_val = min(r.dscr for r in results)

print(f"Minimum DSCR:      {min_dscr_val:.2f}x")
print(f"Covenant Breaches: {breaches}")
print(f"Defaults:          {defaults}")


## 6. I/O vs Full Amortization — DSCR Impact

Interest-only periods improve DSCR significantly during the I/O phase.
This is why borrowers often request I/O periods on value-add deals.


In [ ]:
full_amort_loan = LoanParams(
    loan_amount=8_000_000,
    interest_rate=0.065,
    amortization_years=30,
    loan_term_years=10,
    interest_method="fixed_30_360",
    payments_per_year=12,
)

io_loan = LoanParams(
    loan_amount=8_000_000,
    interest_rate=0.065,
    amortization_years=30,
    loan_term_years=10,
    interest_method="interest_only",
    payments_per_year=12,
)

r_full = dscr.analyze(full_amort_loan, noi, min_dscr=1.25)
r_io   = dscr.analyze(io_loan, noi, min_dscr=1.25)
r_pio  = dscr.analyze(loan, noi, min_dscr=1.25)

print(f"{'Year':<8} {'Full Amort':<15} {'Partial I/O':<15} {'Full I/O'}")
print("-" * 50)
for f, p, i in zip(r_full, r_pio, r_io):
    print(f"  {f.year:<6} {f.dscr:<15.2f} {p.dscr:<15.2f} {i.dscr:.2f}")


## 7. Maximum Loan Sizing

What is the maximum loan this property can support given DSCR, LTV,
and Debt Yield constraints?


In [ ]:
result = sizing.size_loan(
    loan,
    noi=750_000,
    property_value=12_000_000,
    min_dscr=1.25,
    max_ltv=0.75,
    min_debt_yield=0.08,
)


## 8. Rate Shock Stress Test

How does DSCR change if interest rates rise 100, 200, or 300 basis points?


In [ ]:
rate_df = stress.rate_shock(
    loan,
    noi,
    rate_shocks=[0.01, 0.02, 0.03],
    min_dscr=1.25,
)


## 9. NOI Stress Test

What happens to DSCR if NOI declines 10%, 20%, or 30%?


In [ ]:
noi_df = stress.noi_stress(
    loan,
    noi,
    noi_haircuts=[0.10, 0.20, 0.30],
    min_dscr=1.25,
)


## 10. Break-Even NOI

What is the minimum NOI needed to stay above the covenant threshold?


In [ ]:
be_breakeven = stress.break_even_noi(loan, min_dscr=1.0)
be_covenant  = stress.break_even_noi(loan, min_dscr=1.25)

cushion = noi.get_effective_noi(1) - be_covenant
print(f"\nCurrent Effective NOI (Y1): \${noi.get_effective_noi(1):,.0f}")
print(f"Break-even at 1.00x DSCR:   \${be_breakeven:,.0f}")
print(f"Break-even at 1.25x DSCR:   \${be_covenant:,.0f}")
print(f"NOI Cushion above covenant:  \${cushion:,.0f}")


## Summary

This notebook demonstrated the full dscr-tools workflow:

1. **Amortization** — period-by-period schedule with I/O and full amort
2. **Interest method comparison** — 30/360, Actual/360, I/O, Partial I/O
3. **DSCR analysis** — annual tracking with covenant breach detection
4. **I/O impact** — how interest-only periods improve coverage ratios
5. **Loan sizing** — max loan via DSCR, LTV, and Debt Yield constraints
6. **Stress testing** — rate shock and NOI haircut sensitivity tables
7. **Break-even** — minimum NOI to maintain covenant compliance

**GitHub:** https://github.com/Jaypatel1511/dscr-tools
**PyPI:** https://pypi.org/project/dscr-tools
